# 01. LoRA 파인튜닝 — 적은 자원으로 LLM 길들이기

대형 언어모델(LLM)을 통째로 학습하려면 막대한 자원이 필요합니다.
**LoRA(Low-Rank Adaptation)** 는 원본 가중치는 고정하고 작은 추가 행렬만 학습해,
적은 메모리로 효율적으로 파인튜닝하는 표준 기법입니다.

이 노트북에서는 작은 LLM에 LoRA를 적용하고 `trl`의 SFTTrainer로 지도 파인튜닝(SFT)합니다.

1. 모델·토크나이저 로드
2. LoRA 설정
3. 데이터 준비
4. SFTTrainer로 학습

> 교육용으로 작은 모델(TinyLlama)을 사용합니다. 모델은 실행 시 자동 다운로드됩니다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print('GPU:', torch.cuda.is_available())
model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

## 1. 모델·토크나이저 로드

작은 인과적 언어모델(causal LM)을 불러옵니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
print('모델 로드 완료')

## 2. LoRA 설정

`LoraConfig`로 어떤 층에 LoRA를 적용할지, 랭크(`r`)·스케일(`lora_alpha`) 등을 정합니다.
`get_peft_model`로 감싸면 학습 대상 파라미터가 크게 줄어듭니다 (보통 전체의 1% 미만).

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # 학습 대상 비율 출력

## 3. 데이터 준비

지도 파인튜닝용 instruction 데이터셋을 불러옵니다. 학습 속도를 위해 일부만 사용합니다.

In [ ]:
dataset = load_dataset('tatsu-lab/alpaca', split='train')
dataset = dataset.shuffle(seed=42).select(range(500))

def to_text(ex):
    instr, inp, out = ex['instruction'], ex.get('input', ''), ex['output']
    prompt = f"### Instruction:\n{instr}\n"
    if inp:
        prompt += f"### Input:\n{inp}\n"
    prompt += f"### Response:\n{out}"
    return {'text': prompt}

dataset = dataset.map(to_text)
print('예시:\n', dataset[0]['text'][:200])

## 4. SFTTrainer로 학습

`trl`의 SFTTrainer는 Trainer를 확장해 LLM 지도 파인튜닝을 간결하게 해줍니다. GPU에서 수 분 내 끝납니다.

In [ ]:
sft_config = SFTConfig(
    output_dir='/workspace/lora-out',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    max_length=512,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
)
trainer.train()
print('LoRA 파인튜닝 완료')

### 정리

- LoRA로 원본 가중치는 고정하고 작은 추가 행렬만 학습해, 적은 자원으로 LLM을 파인튜닝했습니다.
- `trl` SFTTrainer로 instruction 데이터에 대한 지도 파인튜닝을 수행했습니다.
- 다음 노트북에서는 4비트 양자화를 결합한 **QLoRA**로 더 큰 모델을 더 적은 메모리에서 다룹니다.